# Network Analysis — Streckenveränderungen 2023–2025

Das Zürcher Tramnetz ist kein statisches Objekt.
Im Analysezeitraum 2023–2025 gab es mit dem Fahrplanwechsel Dezember 2023 den größten Netzausbau in der Geschichte der VBZ.
Dieses Notebook untersucht **was** sich verändert hat, **wo** und **wann** — und was das für die Pünktlichkeit bedeutet.

**Zentrale Fragen:**
1. Wo haben sich die meisten Änderungen abgespielt? — Haltestellen und Stadtteile
2. Wieviel hat sich verändert? — Quantifizierung pro Linie und gesamt
3. Wann fanden die Änderungen statt? — Zeitachse
4. Hat sich die Lage nach dem Ausbau verbessert oder verschlechtert? — Einlaufzeit neuer Abschnitte
5. Welche Knotenpunkte sind kritische Hotspots? — Kaskaden und Linienüberschneidungen
6. Welche Stadtteile profitieren? — Versorgungsqualität

**Rückschluss für alle weiteren Analysen:** → am Ende dieses Notebooks

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import folium
from IPython.display import IFrame, display

TRAIN, TEST, lf = setup_analysis("03_analysis_network")
lf_all = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])

%load_ext autoreload
%autoreload 2

In [ ]:
# ── GTFS-Vergleichsdaten laden ────────────────────────────────────────────────
# Quelle: sf_data-research GTFS j23 / j24 / j25
# Stop-Sequenzen wurden in der Netzstruktur-Analyse extrahiert und gecacht.
# Falls noch nicht vorhanden: einmalig aus sf_data-research neu berechnen.

import pandas as pd
from pathlib import Path

SF_GTFS = PATHS["root"].parent / "sf_data-research" / "data" / "raw" / "vbz" / "gtfs"

YEARS = {
    "j23": SF_GTFS / "2023_google_transit",
    "j24": SF_GTFS / "2024_google_transit",
    "j25": SF_GTFS / "2025_google_transit",
}

def get_stops_per_line(year, path):
    """Gibt pro Linie die repräsentativen Haltestellen-Namen zurück (direction 0)."""
    routes = pd.read_csv(path / "routes.txt", dtype=str)
    trips  = pd.read_csv(path / "trips.txt",  dtype=str)
    stops  = pd.read_csv(path / "stops.txt",  dtype=str)
    stops["stop_lat"] = stops["stop_lat"].astype(float)
    stops["stop_lon"] = stops["stop_lon"].astype(float)

    tram_r = routes[
        routes["route_id"].str.startswith("1-") &
        routes["route_short_name"].str.match(r"^\d+$|^E$")
    ][["route_id","route_short_name"]]

    trips_t = (trips.merge(tram_r, on="route_id")
               [lambda df: df["direction_id"] == "0"]
               [["route_short_name","shape_id","trip_id"]])

    rep = (trips_t.groupby(["route_short_name","shape_id"])
           .size().reset_index(name="n")
           .sort_values("n", ascending=False)
           .groupby("route_short_name").first().reset_index())

    result = {}
    trip_ids = [trips_t[trips_t["shape_id"] == sid]["trip_id"].iloc[0]
                for sid in rep["shape_id"].tolist()]

    st_df = (pl.scan_csv(str(path / "stop_times.txt"), infer_schema_length=100)
             .filter(pl.col("trip_id").is_in(trip_ids))
             .sort(["trip_id","stop_sequence"]).collect())

    for _, r in rep.iterrows():
        ln  = r["route_short_name"]
        sid = r["shape_id"]
        mask = trips_t["shape_id"] == sid
        if not mask.any(): continue
        tid = trips_t[mask]["trip_id"].iloc[0]
        stop_ids = st_df.filter(pl.col("trip_id") == tid)["stop_id"].to_list()
        st_m = stops[stops["stop_id"].isin(stop_ids)].copy()
        names = set(st_m["stop_name"].dropna())
        lats  = st_m.set_index("stop_name")["stop_lat"].to_dict()
        lons  = st_m.set_index("stop_name")["stop_lon"].to_dict()
        result[ln] = {"names": names, "coords": {n: (lats.get(n), lons.get(n)) for n in names}}
    return result

log("Lade GTFS j23 / j24 / j25 ...")
gtfs = {yr: get_stops_per_line(yr, path) for yr, path in YEARS.items()}
all_lines = sorted(set(ln for yr in gtfs for ln in gtfs[yr]), key=lambda x: int(x) if x.isdigit() else 99)
success(f"{len(all_lines)} Linien geladen: {all_lines}")

In [ ]:
# ── Änderungsmatrix aufbauen ──────────────────────────────────────────────────
LINE_COLORS = {
    "2":"#E20A16","3":"#00892F","4":"#11296F","5":"#734522","6":"#CA7D3C",
    "7":"#000000","8":"#8AB51F","9":"#11296F","10":"#E12472","11":"#00892F",
    "12":"#92D6E3","13":"#FFCC00","14":"#008DC5","15":"#E20A16","17":"#8E224D",
    "18":"#E20A16","19":"#E20A16","E":"#E20A16",
}

rows = []
for ln in all_lines:
    n23 = gtfs["j23"].get(ln, {}).get("names", set())
    n24 = gtfs["j24"].get(ln, {}).get("names", set())
    n25 = gtfs["j25"].get(ln, {}).get("names", set())
    added_j24    = n24 - n23
    removed_j24  = n23 - n24
    added_j25    = n25 - n24
    removed_j25  = n24 - n25
    rows.append({
        "line": ln, "n_j23": len(n23), "n_j24": len(n24), "n_j25": len(n25),
        "added_j24": len(added_j24), "removed_j24": len(removed_j24),
        "added_j25":  len(added_j25),  "removed_j25":  len(removed_j25),
        "changed_j24": bool(added_j24 or removed_j24),
        "changed_j25": bool(added_j25 or removed_j25),
        "names_j23": n23, "names_j24": n24, "names_j25": n25,
        "names_added_j24": added_j24, "names_removed_j24": removed_j24,
        "names_added_j25": added_j25,  "names_removed_j25":  removed_j25,
        "coords_j23": gtfs["j23"].get(ln, {}).get("coords", {}),
        "coords_j24": gtfs["j24"].get(ln, {}).get("coords", {}),
        "coords_j25": gtfs["j25"].get(ln, {}).get("coords", {}),
    })
changes = pd.DataFrame(rows)
success(f"Änderungsmatrix: {len(changes)} Linien")
show_df(changes[["line","n_j23","n_j24","n_j25","added_j24","removed_j24","added_j25","removed_j25"]].set_index("line"))

## Überblick — Das Netz im Wandel

In [ ]:
section_header("Netzübersicht 2023–2025")

total_j23 = changes["n_j23"].sum()
total_j24 = changes["n_j24"].sum()
total_j25 = changes["n_j25"].sum()
changed_lines_j24 = changes["changed_j24"].sum()
changed_lines_j25 = changes["changed_j25"].sum()
net_added_j24 = changes["added_j24"].sum() - changes["removed_j24"].sum()

print(f"Linien im Netz:  j23={changes['n_j23'].gt(0).sum()} | j24={changes['n_j24'].gt(0).sum()} | j25={changes['n_j25'].gt(0).sum()}")
print(f"Halte gesamt:    j23={total_j23} | j24={total_j24} | j25={total_j25}")
print(f"Linien geändert: j23→j24: {changed_lines_j24} | j24→j25: {changed_lines_j25}")
print(f"Netto neue Halte j23→j24: +{net_added_j24}")

Im Fahrplanwechsel Dezember 2023 (j23 → j24) wurden **{changed_lines_j24} von {n_lines} Linien** verändert.
Der Ausbau ist einseitig: fast alle Änderungen passierten in diesem einen Wechsel.
j24 → j25 ist vergleichsweise stabil.

> Die interaktive Karte mit allen Linien, Haltestellen und Jahresvergleich:
> [`reports/figures/tram_lines_map.html`](../reports/figures/tram_lines_map.html)

### Interaktive Karte

In [ ]:
# Karte direkt im Notebook anzeigen
map_path = PATHS["root"] / "reports" / "figures" / "tram_lines_map.html"
display(IFrame(src=str(map_path), width="100%", height="600px"))

## Wo? — Räumliche Verteilung der Änderungen

In [ ]:
section_header("Wo haben sich die meisten Änderungen abgespielt?")

# ── 1. Karte: neue und entfernte Halte ───────────────────────────────────────
m = folium.Map(location=[47.378, 8.540], zoom_start=13, tiles="CartoDB positron")

for _, row in changes.iterrows():
    ln    = row["line"]
    color = LINE_COLORS.get(ln, "#888")

    # Neue Halte (hinzugekommen j24)
    for name, (lat, lon) in row["coords_j24"].items():
        if name in row["names_added_j24"] and lat and lon:
            folium.CircleMarker(
                location=[lat, lon], radius=6,
                color="#FF6B00", weight=1.5, fill=True, fill_color="#FF6B00", fill_opacity=0.85,
                tooltip=f"<b>{name}</b><br>Linie {ln} — neu ab j24"
            ).add_to(m)

    # Entfernte Halte (nur j23)
    for name, (lat, lon) in row["coords_j23"].items():
        if name in row["names_removed_j24"] and lat and lon:
            folium.CircleMarker(
                location=[lat, lon], radius=5,
                color="#aaa", weight=1, fill=True, fill_color="#ddd", fill_opacity=0.7,
                tooltip=f"<b>{name}</b><br>Linie {ln} — nur j23"
            ).add_to(m)

# Legende
legend_html = """
<div style="position:fixed;bottom:20px;left:12px;z-index:1000;background:white;
     padding:9px 12px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.15);font-size:11px">
  <div style="display:flex;align-items:center;gap:7px;margin:3px 0">
    <div style="width:10px;height:10px;background:#FF6B00;border-radius:50%"></div>Neu ab Dez 2023
  </div>
  <div style="display:flex;align-items:center;gap:7px;margin:3px 0">
    <div style="width:10px;height:10px;background:#ddd;border:1px solid #aaa;border-radius:50%"></div>Entfernt nach j23
  </div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

map_out = PATHS["figures"] / "network_changes_map.html"
m.save(str(map_out))
display(IFrame(src=str(map_out), width="100%", height="500px"))

In [ ]:
# ── 2. Neue Halte nach Stadtkreis ─────────────────────────────────────────────
# Stadtkreis-Zuordnung via Master-Daten (stop_name → district)
district_lookup = (
    lf_all
    .select(["stop_name", "district_nr", "district_name"])
    .drop_nulls()
    .unique()
    .collect()
    .to_pandas()
    .set_index("stop_name")
)

# Alle neuen Halte j24 über alle Linien sammeln
all_new = {}
for _, row in changes.iterrows():
    for name in row["names_added_j24"]:
        all_new[name] = row["line"]

new_df = pd.DataFrame({"stop_name": list(all_new.keys()), "line": list(all_new.values())})
new_df = new_df.merge(district_lookup.reset_index(), on="stop_name", how="left")

by_district = (new_df.groupby("district_name")["stop_name"]
               .nunique().reset_index(name="new_stops")
               .sort_values("new_stops", ascending=False)
               .dropna())

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(by_district["district_name"], by_district["new_stops"],
               color=cfg.palette_n(len(by_district)))
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_xlabel("Anzahl neuer Haltestellen")
ax.set_title("Neue Haltestellen ab Dez 2023 — nach Stadtkreis", fontweight="bold")
plt.tight_layout()
plt.show()

**Beobachtung:**

Die räumliche Verteilung zeigt klar, welche Stadtteile durch den Fahrplanwechsel Dezember 2023 neu erschlossen oder besser vernetzt wurden.
Die neuen Halte konzentrieren sich auf bestimmte Stadtkreise — das sind die Gebiete, in denen Linien 9, 11 und 13 ihre Strecken massiv ausgebaut haben.

Für die weitere Analyse bedeutet das: Verspätungsmuster in diesen Stadtkreisen nach j24 sind nicht direkt mit j23 vergleichbar — es gibt schlicht mehr Messpunkte und andere Streckenlängen.

## Wieviel? — Quantifizierung der Änderungen

In [ ]:
section_header("Quantifizierung: Halte pro Linie und Jahr")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Links: Halte pro Linie, alle drei Jahre ──────────────────────────────────
lines_with_data = changes[changes[["n_j23","n_j24","n_j25"]].max(axis=1) > 0]
x = np.arange(len(lines_with_data))
w = 0.27
colors = cfg.palette_n(3)

ax = axes[0]
ax.bar(x - w, lines_with_data["n_j23"], w, label="2023", color=colors[0], alpha=0.85)
ax.bar(x,     lines_with_data["n_j24"], w, label="2024", color=colors[1], alpha=0.85)
ax.bar(x + w, lines_with_data["n_j25"], w, label="2025", color=colors[2], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f"L{ln}" for ln in lines_with_data["line"]], fontsize=9)
ax.set_ylabel("Anzahl Haltestellen")
ax.set_title("Haltestellen pro Linie — 2023 / 2024 / 2025", fontweight="bold")
ax.legend()

# ── Rechts: Netto-Änderung j23→j24 pro Linie ────────────────────────────────
ax2 = axes[1]
changed = changes[changes["added_j24"] + changes["removed_j24"] > 0].copy()
changed["net"] = changed["added_j24"] - changed["removed_j24"]
changed = changed.sort_values("net", ascending=True)
bar_colors = [LINE_COLORS.get(ln, "#888") for ln in changed["line"]]
bars = ax2.barh(changed["line"], changed["net"], color=bar_colors, edgecolor="white", linewidth=0.5)
ax2.axvline(0, color="#999", linewidth=0.8, linestyle="--")
ax2.bar_label(bars, labels=[f"+{v}" if v > 0 else str(v) for v in changed["net"]], padding=3, fontsize=9)
ax2.set_xlabel("Netto neue Haltestellen (j23 → j24)")
ax2.set_title("Netto-Änderung je Linie — Fahrplanwechsel Dez 2023", fontweight="bold")

plt.tight_layout()
plt.show()

**Beobachtung:**

Linie 13 ist der extremste Fall: von 11 auf 30 Haltestellen — eine Verdreifachung.
Linien 9 (+8) und 11 (+13) folgen. Linien 10, 12, 14 und 17 sind über alle drei Jahre stabil.

Das Netz ist nach j24 deutlich größer — aber die Änderungen sind auf drei Linien konzentriert.
Die stabilen Linien (10, 12, 14, 17) können ohne Einschränkung jahresübergreifend verglichen werden.

## Wann? — Zeitachse der Änderungen

In [ ]:
section_header("Zeitachse: Fahrplanwechsel und Auswirkungen auf Delay")

# Monatliche Durchschnittsverspätung für die drei stark geänderten Linien
# Vergleich mit den stabilen Linien als Referenz

changed_lines_list = ["9", "11", "13"]
stable_lines_list  = ["10", "12", "17"]
FAHRPLANWECHSEL = "2024-01"   # Dez 2023 → gilt ab Jan 2024

monthly = (
    lf_all
    .filter(~pl.col("canceled"))
    .with_columns([
        pl.col("operating_date").dt.strftime("%Y-%m").alias("month"),
        pl.col("line_name").cast(pl.Utf8).alias("line"),
    ])
    .filter(pl.col("line").is_in(changed_lines_list + stable_lines_list))
    .group_by(["month","line"])
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"))
    .sort(["line","month"])
    .collect()
    .to_pandas()
)
monthly["month"] = pd.to_datetime(monthly["month"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, lines, title in [
    (axes[0], changed_lines_list, "Stark geänderte Linien (9 · 11 · 13)"),
    (axes[1], stable_lines_list,  "Stabile Linien — Referenz (10 · 12 · 17)"),
]:
    for ln in lines:
        sub = monthly[monthly["line"] == ln]
        ax.plot(sub["month"], sub["avg_delay"],
                label=f"Linie {ln}", color=LINE_COLORS.get(ln, "#888"),
                linewidth=2, marker="o", markersize=3)
    ax.axvline(pd.Timestamp("2024-01-01"), color="#E20A16", linewidth=1.5,
               linestyle="--", alpha=0.8, label="Fahrplanwechsel Dez 2023")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Ø Ankunftsverspätung (s)")
    ax.legend(fontsize=9)
    ax.set_xlim(pd.Timestamp("2023-01-01"), pd.Timestamp("2025-11-30"))

plt.suptitle("Monatliche Durchschnittsverspätung — vor und nach Fahrplanwechsel", fontweight="bold")
plt.tight_layout()
plt.show()

**Beobachtung:**

Der Fahrplanwechsel ist als vertikale Linie markiert. Für Linien 9, 11 und 13 sehen wir
den Verlauf vor der Streckenerweiterung (2023) und nach der Umstrukturierung (2024–2025).

Die stabilen Linien 10, 12 und 17 dienen als **Referenz-Baseline**: Veränderungen dort
sind auf externe Faktoren (Wetter, Events, saisonale Muster) zurückzuführen — nicht auf Netzänderungen.
Ein systematischer Unterschied zwischen geänderten und stabilen Linien nach dem Wechsel wäre ein starkes Signal.

## Einlaufzeit — Performen neue Abschnitte anders?

In [ ]:
section_header("Einlaufzeit: Neue Haltestellen ab j24")

# Für neue Halte (nur in j24/j25): monatliche Verspätung ab Jan 2024
# Vergleich mit bestehenden Halten der gleichen Linie

# Sammle neue Halte-Namen
new_stop_names = set()
for _, row in changes.iterrows():
    new_stop_names.update(row["names_added_j24"])

monthly_stops = (
    lf_all
    .filter(~pl.col("canceled"))
    .filter(pl.col("operating_date") >= pl.lit("2024-01-01").str.to_date())
    .with_columns([
        pl.col("operating_date").dt.strftime("%Y-%m").alias("month"),
        pl.col("stop_name").cast(pl.Utf8).alias("stop"),
        pl.col("line_name").cast(pl.Utf8).alias("line"),
        pl.col("stop_name").cast(pl.Utf8).is_in(list(new_stop_names)).alias("is_new"),
    ])
    .filter(pl.col("line").is_in(["9","11","13"]))
    .group_by(["month","line","is_new"])
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"),
         pl.col("stop").n_unique().alias("n_stops"))
    .sort(["line","month"])
    .collect()
    .to_pandas()
)
monthly_stops["month"] = pd.to_datetime(monthly_stops["month"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ln in zip(axes, ["9","11","13"]):
    sub = monthly_stops[monthly_stops["line"] == ln]
    for is_new, label, ls in [(True,"Neue Halte (ab j24)","--"), (False,"Bestehende Halte","-")]:
        s = sub[sub["is_new"] == is_new]
        if not s.empty:
            ax.plot(s["month"], s["avg_delay"],
                    label=label, linewidth=2, linestyle=ls,
                    color=LINE_COLORS.get(ln, "#888"),
                    alpha=1.0 if not is_new else 0.6,
                    marker="o", markersize=3)
    ax.set_title(f"Linie {ln}", fontweight="bold")
    ax.set_ylabel("Ø Ankunftsverspätung (s)")
    ax.legend(fontsize=9)

plt.suptitle("Einlaufzeit: Neue vs. bestehende Haltestellen ab Jan 2024", fontweight="bold")
plt.tight_layout()
plt.show()

**Beobachtung:**

Neue Haltestellen (gestrichelt) vs. bestehende Haltestellen (durchgezogen) auf denselben Linien.
Falls neue Halte in den ersten Monaten deutlich mehr Verspätung zeigen, ist das ein Hinweis auf eine Einlaufzeit —
Fahrgäste, Fahrer und Betrieb brauchen Zeit, sich auf die neuen Abschnitte einzustellen.

## Hotspots & Kaskaden — Kritische Knotenpunkte

In [ ]:
section_header("Hotspots: Haltestellen mit den meisten Linien")

# Welche Haltestellen werden in j25 von den meisten verschiedenen Linien bedient?
stop_lines = {}
for _, row in changes.iterrows():
    ln = row["line"]
    for name in row["names_j25"]:
        stop_lines.setdefault(name, set()).add(ln)

hotspot_df = (pd.DataFrame({
    "stop_name": list(stop_lines.keys()),
    "n_lines":   [len(v) for v in stop_lines.values()],
    "lines":     [", ".join(sorted(v, key=lambda x: int(x) if x.isdigit() else 99))
                  for v in stop_lines.values()]
})
.sort_values("n_lines", ascending=False)
.head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Top Hotspots
ax = axes[0]
bars = ax.barh(hotspot_df["stop_name"][:15][::-1],
               hotspot_df["n_lines"][:15][::-1],
               color=cfg.palette_n(1)[0])
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel("Anzahl Linien")
ax.set_title("Top 15 Haltestellen nach Linienanzahl (j25)", fontweight="bold")

# Scatter: Linienanzahl vs. durchschn. Verspätung
delay_per_stop = (
    lf_all
    .filter(~pl.col("canceled"))
    .filter(pl.col("operating_date") >= pl.lit("2024-01-01").str.to_date())
    .group_by("stop_name")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"),
         pl.len().alias("n_obs"))
    .filter(pl.col("n_obs") > 1000)
    .collect()
    .to_pandas()
)

hotspot_merged = hotspot_df.merge(delay_per_stop, on="stop_name", how="inner")

ax2 = axes[1]
scatter = ax2.scatter(hotspot_merged["n_lines"], hotspot_merged["avg_delay"],
                      s=hotspot_merged["n_obs"]/500, alpha=0.7,
                      color=cfg.palette_n(1)[0], edgecolors="white", linewidth=0.5)
for _, r in hotspot_merged[hotspot_merged["n_lines"] >= 4].iterrows():
    ax2.annotate(r["stop_name"], (r["n_lines"], r["avg_delay"]),
                 fontsize=8, ha="left", va="bottom",
                 xytext=(4, 4), textcoords="offset points")
ax2.set_xlabel("Anzahl Linien an der Haltestelle")
ax2.set_ylabel("Ø Ankunftsverspätung (s)")
ax2.set_title("Linienanzahl vs. Verspätung — Knotenpunkte 2024–2025", fontweight="bold")
ax2.axhline(delay_per_stop["avg_delay"].median(), color="#999", linestyle="--",
            linewidth=1, label=f"Median {delay_per_stop['avg_delay'].median():.0f}s")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

**Beobachtung:**

Die zentralen Knotenpunkte (Paradeplatz, Bellevue, HB-Bereich) werden von den meisten Linien bedient.
Der Scatter zeigt ob Haltestellen mit mehr Linien auch mehr Verspätung akkumulieren.

Falls ja: diese Knotenpunkte sind systemische Risikopunkte — eine Störung dort pflanzt sich auf alle durchfahrenden Linien fort (*Kaskadeneffekt*).
Das sind die wichtigsten Haltestellen für das Prediction-Modell.

## Versorgungsqualität — Welche Stadtteile profitieren?

In [ ]:
section_header("Versorgungsqualität: Neue Erschliessung nach Stadtkreis")

# Vor/Nach-Vergleich: Anzahl Linien pro Stadtkreis in j23 vs. j25
# Proxy: wie viele unterschiedliche Linien fahren Halte in diesem Kreis an?

def lines_per_district(year_key):
    lf_year = lf_all if year_key == "j25" else lf_all  # nutze Master-Daten, gefiltert nach Jahr
    yr_start = {"j23":"2023-01-01","j24":"2024-01-01","j25":"2025-01-01"}[year_key]
    yr_end   = {"j23":"2023-12-31","j24":"2024-12-31","j25":"2025-11-30"}[year_key]
    return (
        lf_all
        .filter(pl.col("operating_date").is_between(
            pl.lit(yr_start).str.to_date(), pl.lit(yr_end).str.to_date()))
        .filter(pl.col("district_nr").is_not_null())
        .select(["district_name","line_name"])
        .unique()
        .group_by("district_name")
        .agg(pl.col("line_name").n_unique().alias(f"lines_{year_key}"))
        .collect().to_pandas()
    )

dist_j23 = lines_per_district("j23")
dist_j25 = lines_per_district("j25")
dist_cmp = dist_j23.merge(dist_j25, on="district_name", how="outer").fillna(0)
dist_cmp["delta"] = dist_cmp["lines_j25"] - dist_cmp["lines_j23"]
dist_cmp = dist_cmp.sort_values("delta", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ["#E20A16" if d < 0 else "#00892F" if d > 0 else "#aaa" for d in dist_cmp["delta"]]
bars = ax.barh(dist_cmp["district_name"], dist_cmp["delta"], color=colors_bar, edgecolor="white")
ax.bar_label(bars, labels=[f"+{v:.0f}" if v > 0 else f"{v:.0f}" for v in dist_cmp["delta"]],
             padding=3, fontsize=9)
ax.axvline(0, color="#999", linewidth=0.8)
ax.set_xlabel("Δ Anzahl Linien (2025 vs. 2023)")
ax.set_title("Veränderung der Linienanbindung nach Stadtkreis — 2023 → 2025", fontweight="bold")
plt.tight_layout()
plt.show()

**Beobachtung:**

Positiver Delta (grün): Stadtkreise die durch den Netzausbau mehr Tramlinien bekommen haben.
Negativer Delta (rot): Stadtkreise mit weniger Linienanbindung — z.B. durch Streckenanpassungen.

Das zeigt direkt welche Gebiete vom Ausbau profitiert haben und welche möglicherweise schlechter dastehen.

## Fazit & Rückschluss auf die weitere Analyse

### Was die Netzanalyse für alle weiteren Notebooks bedeutet

Die GTFS-Analyse über 2023, 2024 und 2025 ergibt folgende strukturelle Erkenntnisse:

#### Stabiles Teilnetz — uneingeschränkt vergleichbar
Linien **10, 12, 14, 17** haben identische Streckenführungen über alle drei Jahre.
Für diese Linien können Jahresvergleiche direkt und ohne Einschränkung gezogen werden.

#### Verändertes Teilnetz — Jahresvergleich mit Vorbehalt
| Linie | j23 | j24 | j25 | Kontext |
| :---: | ---: | ---: | ---: | :--- |
| **9** | 24 Halte | 32 Halte | 32 Halte | +8 neue Abschnitte ab Dez 2023 |
| **11** | 20 Halte | 33 Halte | 34 Halte | +13 neue Abschnitte ab Dez 2023 |
| **13** | 11 Halte | 30 Halte | 30 Halte | +19 Halte (+173%) ab Dez 2023 |

Für diese Linien ist Linie 9 in 2023 strukturell eine **andere** Linie als Linie 9 in 2024–2025.
Direkte Jahresvergleiche sind möglich, müssen aber im Kontext des Streckenausbaus interpretiert werden.

#### Das `gtfs_year`-Feature
Ein neues binäres Feature `gtfs_year` (Werte: `j23` | `j24_j25`) kodiert den Strukturbruch im Dezember 2023.
Es soll in `02_preparation.ipynb` als Feature Engineering-Schritt eingebaut werden.

```python
# In 02_preparation.ipynb hinzufügen:
pl.when(pl.col("operating_date") < pl.lit("2024-01-01").str.to_date())
  .then(pl.lit("j23"))
  .otherwise(pl.lit("j24_j25"))
  .alias("gtfs_year")
```

**Bedeutung:** Für die Linien 9, 11 und 13 trägt `gtfs_year` implizit Information über die
Streckenlänge und das Betriebsmuster. Für die stabilen Linien ist es eine Proxy-Variable für allgemeine Zeittrends.

#### Hinweis für alle Analysis-Notebooks
> Alle Analysen in `03_analysis_spatial`, `03_analysis_temporal`, `03_analysis_weather` und `03_analysis_events`
> sollten bei Linien-bezogenen Befunden den Netzwechsel Dezember 2023 als Kontextinformation nennen.
> Detaillierte Aufschlüsselung immer mit Verweis auf dieses Notebook: `03_analysis_network.ipynb`.